In [7]:
from backtest_functions.backtest_functions import BacktestFunctions as bf
from strategy.strategy import Strategy
import pandas as pd
from tqdm import tqdm
from database.adatabase import ADatabase

In [8]:
db = ADatabase("sapling")
market = ADatabase("market")
market.connect()
tickers = market.retrieve("russell1000")["ticker"].values
market.disconnect()
names = Strategy._member_names_
queries = []
for name in tqdm(names):
    for holding_period in [5,10,15]:
        for ascending in [True,False]:
            for positions in [5]:
                for stop_loss in [0.01,0.05]:
                    query = {
                        "strategy":name,
                        "holding_period":15,
                        "positions":positions,
                        "stop_loss":stop_loss,
                        "ascending":ascending,
                        "tickers":tickers
                    }
                    queries.append(query)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 14979.66it/s]


In [9]:
query

{'strategy': 'TECHNICAL',
 'holding_period': 15,
 'positions': 5,
 'stop_loss': 0.05,
 'ascending': False,
 'tickers': array(['TXG', 'MMM', 'ABT', ..., 'ZM', 'ZI', 'ZS'], dtype=object)}

In [ ]:
analysis = []
db.connect()
db.drop("kpi_minute")
for query in tqdm(queries):
    try:
        results = bf.backtest_minute(query)
        db.store("kpi_minute",pd.DataFrame([results["kpi"] | query]).drop("tickers",axis=1))
    except Exception as e:
        print(str(e))
        continue
db.disconnect()

 47%|█████████████████████████████████████████████████████████████▊                                                                     | 85/180 [2:00:30<2:59:21, 113.28s/it]